In [53]:
from langchain_chroma import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

### Step 1a - indexing (Document Ingestion)

In [8]:
ytt_api = YouTubeTranscriptApi()

In [30]:
video_id = "Gfr50f6ZBvo"
try:
    transcript_list = ytt_api.fetch(video_id)
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)
    
except TranscriptsDisabled:
    print("No captions available for this video.")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

### Step 1b - Indexing (Text Splitting)

In [31]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

In [32]:
chunks = text_splitter.create_documents([transcript])

In [33]:
len(chunks)

168

In [34]:
chunks

[Document(metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to inte

### 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [35]:
vector_store = Chroma.from_documents(
    embedding=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"),
    persist_directory="my_chroma_db",
    collection_name="sample",
    documents=chunks
)

In [36]:
vector_store.get()

{'ids': ['1dac489d-c63c-47f4-a8f8-71dce7912f9f',
  'ed508908-175b-4a92-94e8-4f3122e44dec',
  '853fe312-470f-4e5b-93e6-f656a531da3f',
  '6106e53e-f72e-43cb-8945-f2103ad5c6f5',
  'e4b26dc8-f348-4436-a207-e363f429a03b',
  '91b32cb1-11e3-4d87-b765-b3e95f991f9b',
  '5be55545-ab50-4527-a142-e4b80ebf1ccc',
  'b8e928b6-13fd-43f9-a42d-1155dfc68a07',
  '29615139-16b8-465b-8596-796e302af5de',
  'e1d3507e-45cb-4803-97fb-e5e8ec0f5b96',
  'bccc7891-6e41-470d-8939-29ff433dd5b7',
  '5f4e1421-2a30-488d-9bfe-0ecd6fdcd8db',
  '39675391-32aa-405e-89de-2137ea83ea8f',
  'cef6ee70-aa46-44d0-b4af-ac022c41cd18',
  'f8c84e4c-108b-4851-8678-971e62d4d1bb',
  '47ae3022-714e-4399-8090-ce575f1e8bcf',
  '8b07d579-188a-44c1-8fa0-dfc6ca55c065',
  'eb8a98bb-401f-40b9-8bcc-239f71de7cd1',
  '5b4321e5-9f7f-472e-842b-c433580b4e70',
  '2d927aab-26a6-4ea8-a798-2094104e7ad4',
  '1bda6224-5944-4959-bb16-e95510b2a14b',
  '054dd352-4754-4308-af06-c72a71d68e3c',
  '41bbf8ab-5591-4ab2-8861-831427611567',
  '03fc60e4-583d-4a49-9c60-

### Step 2 - Retrival

In [38]:
retriver = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [39]:
retriver

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x000001B2A4AA6FD0>, search_kwargs={'k': 4})

In [41]:
retriver.invoke("what is deepmind")

[Document(id='e0b43eb7-55f0-4584-859c-7ffa58ec1e3f', metadata={}, page_content='i used to discuss um uh uh what were the sort of founding tenets of deep mind and it was very various things one was um algorithmic advances so deep learning you know jeff hinton and cohen just had just sort of invented that in academia but no one in industry knew about it uh we love reinforcement learning we thought that could be scaled up but also understanding about the human brain had advanced um quite a lot uh in the decade prior with fmri machines and other things so we could get some good hints about architectures and algorithms and and sort of um representations maybe that the brain uses so as at a systems level not at a implementation level um and then the other big things were compute and gpus right so we could see a compute was going to be really useful and it got to a place where it became commoditized mostly through the games industry and and that could be taken advantage of and then the final 

### Step 3 - Augmentation

In [42]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [43]:
prompt = PromptTemplate(
    template="""You are a helpful assistant. Answer ONLY from the provided transcript context.
    If the context is insufficient, say 'I don't know'.
    
    {context}
    Question: {question}
    """,
    input_variables=["context", "question"]
)

In [44]:
question = "Is the topic of aliens discussed in the video? If yes then provide details."

In [45]:
retrived_docs = retriver.invoke(question)

In [46]:
context_text = "\n\n".join([doc.page_content for doc in retrived_docs])

In [47]:
context_text

"thoughts it could be some interactions with our mind that we think are originating from us is actually something that uh is coming from other life forms elsewhere consciousness itself might be that it could be but i don't see any sensible argument to the why why would all of the alien species be using this way yes some of them will be more primitive they would be close to our level you know there would there should be a whole sort of normal distribution of these things right some would be aggressive some would be you know curious others would be very stoical and philosophical because you know maybe they're a million years older than us but it's not it shouldn't be like what i mean one one alien civilization might be like that communicating thoughts and others but i don't see why you know potentially the hundreds there should be would be uniform in this way right it could be a violent dictatorship that the the people the alien civilizations that uh become successful become um [Music]\n

In [48]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [49]:
final_prompt

StringPromptValue(text="You are a helpful assistant. Answer ONLY from the provided transcript context.\n    If the context is insufficient, say 'I don't know'.\n\n    thoughts it could be some interactions with our mind that we think are originating from us is actually something that uh is coming from other life forms elsewhere consciousness itself might be that it could be but i don't see any sensible argument to the why why would all of the alien species be using this way yes some of them will be more primitive they would be close to our level you know there would there should be a whole sort of normal distribution of these things right some would be aggressive some would be you know curious others would be very stoical and philosophical because you know maybe they're a million years older than us but it's not it shouldn't be like what i mean one one alien civilization might be like that communicating thoughts and others but i don't see why you know potentially the hundreds there sho

### Step 4 - Generation

In [50]:
answer = llm.invoke(final_prompt)
answer

AIMessage(content='Yes, the topic of aliens is discussed in the transcript.\n\nDetails include:\n*   **Origin of Thoughts:** It\'s considered that some thoughts we perceive as our own might originate from other life forms or alien species.\n*   **Lack of Evidence:** The speaker notes that if alien civilizations were abundant and reached a "space age," we should have heard a "cacophony of voices," but instead, "we heard nothing."\n*   **Arguments for No Contact:** Some argue that we haven\'t done an exhaustive search, are looking in the wrong bands, using the wrong devices, or wouldn\'t recognize an alien form due to its difference.\n*   **"Safari View":** Another argument suggests that humanity is still a primitive, non-space-faring species, and there\'s a universal rule not to interfere with such civilizations.\n*   **Speaker\'s Personal Opinion:** The speaker\'s current feeling, based on available evidence and extensive thought, is that "we are alone" and that this is "the most likel

In [51]:
answer.content

'Yes, the topic of aliens is discussed in the transcript.\n\nDetails include:\n*   **Origin of Thoughts:** It\'s considered that some thoughts we perceive as our own might originate from other life forms or alien species.\n*   **Lack of Evidence:** The speaker notes that if alien civilizations were abundant and reached a "space age," we should have heard a "cacophony of voices," but instead, "we heard nothing."\n*   **Arguments for No Contact:** Some argue that we haven\'t done an exhaustive search, are looking in the wrong bands, using the wrong devices, or wouldn\'t recognize an alien form due to its difference.\n*   **"Safari View":** Another argument suggests that humanity is still a primitive, non-space-faring species, and there\'s a universal rule not to interfere with such civilizations.\n*   **Speaker\'s Personal Opinion:** The speaker\'s current feeling, based on available evidence and extensive thought, is that "we are alone" and that this is "the most likely scenario."\n*   

### Building a Chain

In [54]:
def format_doc(retrived_text):
    context_text = "\n\n".join([doc.page_content for doc in retrived_text])
    return context_text

In [57]:
parallel_chain = RunnableParallel({
    "context" : retriver | RunnableLambda(format_doc),
    "question": RunnablePassthrough()
})

In [58]:
parser = StrOutputParser()

In [59]:
main_chain = parallel_chain | prompt | llm | parser

In [60]:
main_chain.invoke(question)

'Yes, the topic of aliens is discussed in the transcript.\n\nHere are the details:\n\n*   **Potential Nature of Alien Interaction:** It\'s considered that thoughts originating from us might actually be coming from other life forms elsewhere. However, the speaker questions why all alien species would communicate in this uniform way, suggesting there should be a "normal distribution" of characteristics (some primitive, some aggressive, curious, stoical, or philosophical).\n*   **Evidence of Alien Life:** The idea that if aliens were prevalent, we should have heard a "cacophony of voices" or seen evidence like Dyson\'s spheres or blinking suns.\n*   **Reasons for Not Finding Aliens:** Arguments mentioned include not conducting an exhaustive search, looking in the wrong bands or with the wrong devices, or not recognizing what a truly different alien form would be like. Another perspective is the "safari view," where we are seen as a primitive species, and there\'s a universal rule not to i

In [61]:
main_chain.invoke("Can you summarize the video in brief?")


"The video discusses the challenges of understanding why large language models make certain predictions, despite their fluency, and offers resources for learning about transformers and attention. It also touches on the need for more fundamental explanations in physics, beyond the standard model, and the difficulty of simulating large materials in chemistry by approximating Schrödinger's equation to understand electron behavior."